In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import yfinance as yf

In [2]:
weights = np.ones(3)/3

In [3]:
weights

array([0.33333333, 0.33333333, 0.33333333])

In [4]:
L = np.zeros((3, 10))
L

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import yfinance as yf

class AdaptiveLaguerreFilter:
    """自适应拉盖尔滤波器实现"""
    
    def __init__(self, filter_order=3, lambda_param=0.5, learning_rate=0.01):
        """
        初始化滤波器
        
        参数:
            filter_order: 滤波器阶数
            lambda_param: 拉盖尔尺度参数(0-1)
            learning_rate: 学习率
        """
        self.order = filter_order
        self.lambda_param = lambda_param
        self.learning_rate = learning_rate
        self.weights = np.ones(filter_order) / filter_order
        
    def laguerre_filter(self, price_series):
        """计算拉盖尔滤波器系数"""
        n = len(price_series)
        L = np.zeros((self.order, n))
        
        # 第0阶拉盖尔滤波器
        for k in range(n):
            L[0, k] = np.sqrt(1 - self.lambda_param) * (self.lambda_param ** k)
        
        # 高阶拉盖尔滤波器（递归计算）
        for i in range(1, self.order):
            for k in range(n):
                if k == 0:
                    L[i, k] = L[i-1, k] * (1 - self.lambda_param)
                else:
                    L[i, k] = L[i, k-1] * self.lambda_param + L[i-1, k] - L[i-1, k-1] * self.lambda_param
        
        return L
    
    def adaptive_filter(self, prices):
        """自适应滤波处理"""
        n = len(prices)
        filtered_prices = np.zeros(n)
        errors = np.zeros(n)
        
        # 初始预测使用简单移动平均
        filtered_prices[:self.order] = np.mean(prices[:self.order])
        
        for t in range(self.order, n):
            # 使用当前权重进行预测
            prediction = np.dot(self.weights, prices[t-self.order:t])
            filtered_prices[t] = prediction
            
            # 计算预测误差
            error = prices[t] - prediction
            errors[t] = error
            
            # 自适应调整权重（LMS算法）
            if t < n-1:
                input_vector = prices[t-self.order:t]
                self.weights += 2 * self.learning_rate * error * input_vector
                
                # 权重归一化
                self.weights /= np.sum(self.weights)
        
        return filtered_prices, errors

class MomentumStrategy:
    """动量策略实现"""
    
    def __init__(self, filter_order=3, lambda_param=0.7, momentum_period=20):
        self.filter = AdaptiveLaguerreFilter(filter_order, lambda_param)
        self.momentum_period = momentum_period
        self.signals = []
        
    def generate_signals(self, prices):
        """生成交易信号"""
        # ALF滤波处理
        filtered_prices, errors = self.filter.adaptive_filter(prices)
        
        # 计算动量指标
        momentum = np.zeros(len(prices))
        signals = np.zeros(len(prices))  # 0:无仓位, 1:做多, -1:做空
        
        for i in range(self.momentum_period, len(prices)):
            if i >= self.momentum_period:
                # 动量计算：当前滤波价格与momentum_period期前的比率
                momentum[i] = (filtered_prices[i] / filtered_prices[i-self.momentum_period] - 1) * 100
                
                # 生成交易信号
                if momentum[i] > 2.0:  # 动量强劲，做多
                    signals[i] = 1
                elif momentum[i] < -2.0:  # 动量疲软，做空
                    signals[i] = -1
                else:  # 动量中性，保持现状
                    signals[i] = signals[i-1] if i > 0 else 0
        
        return filtered_prices, momentum, signals

def backtest_strategy(symbol, start_date, end_date, initial_capital=100000):
    """策略回测函数"""
    # 获取数据
    data = yf.download(symbol, start=start_date, end=end_date)
    prices = data['Close'].values
    
    # 创建策略实例
    strategy = MomentumStrategy(filter_order=4, lambda_param=0.8, momentum_period=15)
    
    # 生成信号
    filtered_prices, momentum, signals = strategy.generate_signals(prices)
    
    # 执行回测
    capital = initial_capital
    position = 0
    returns = [0]
    portfolio_values = [initial_capital]
    trades = []
    
    for i in range(1, len(prices)):
        # 计算当日收益率
        if position == 1:  # 做多
            daily_return = (prices[i] / prices[i-1] - 1)
        elif position == -1:  # 做空
            daily_return = (prices[i-1] / prices[i] - 1)
        else:  # 无仓位
            daily_return = 0
        
        # 更新资金曲线
        capital *= (1 + daily_return)
        portfolio_values.append(capital)
        returns.append(daily_return)
        
        # 检查信号变化
        if i > 0 and signals[i] != signals[i-1]:
            position = signals[i]
            trades.append(i)
    
    # 计算性能指标
    returns = np.array(returns)
    total_return = (portfolio_values[-1] / initial_capital - 1) * 100
    sharpe_ratio = np.mean(returns) / np.std(returns) * np.sqrt(252) if np.std(returns) > 0 else 0
    max_drawdown = calculate_max_drawdown(portfolio_values)
    
    return {
        'prices': prices,
        'filtered_prices': filtered_prices,
        'momentum': momentum,
        'signals': signals,
        'portfolio_values': portfolio_values,
        'total_return': total_return,
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'trades': trades
    }

def calculate_max_drawdown(portfolio_values):
    """计算最大回撤"""
    peak = portfolio_values[0]
    max_dd = 0
    
    for value in portfolio_values:
        if value > peak:
            peak = value
        dd = (peak - value) / peak
        if dd > max_dd:
            max_dd = dd
    
    return max_dd * 100

def plot_results(results, symbol):
    """绘制回测结果"""
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10))
    
    # 价格和信号图
    ax1.plot(results['prices'], label='Actual Price', linewidth=1)
    ax1.plot(results['filtered_prices'], label='ALF Filtered Price', linewidth=1.5)
    ax1.set_ylabel('Price')
    ax1.set_title(f'ALF Filter Momentum Strategy - {symbol}')
    ax1.legend()
    ax1.grid(True)
    
    # 标记交易信号
    buy_signals = [i for i, sig in enumerate(results['signals']) if sig == 1]
    sell_signals = [i for i, sig in enumerate(results['signals']) if sig == -1]
    
    ax1.scatter(buy_signals, [results['prices'][i] for i in buy_signals], 
               color='green', marker='^', s=100, label='Buy Signal', alpha=0.7)
    ax1.scatter(sell_signals, [results['prices'][i] for i in sell_signals],
               color='red', marker='v', s=100, label='Sell Signal', alpha=0.7)
    
    # 动量指标图
    ax2.plot(results['momentum'], label='Momentum Indicator', color='orange')
    ax2.axhline(y=2, color='red', linestyle='--', alpha=0.7, label='Buy Threshold')
    ax2.axhline(y=-2, color='green', linestyle='--', alpha=0.7, label='Sell Threshold')
    ax2.set_ylabel('Momentum (%)')
    ax2.legend()
    ax2.grid(True)
    
    # 资金曲线图
    ax3.plot(results['portfolio_values'], label='Portfolio Value', color='blue')
    ax3.set_ylabel('Portfolio Value ($)')
    ax3.set_xlabel('Days')
    ax3.legend()
    ax3.grid(True)
    
    plt.tight_layout()
    plt.show()

# 示例使用
if __name__ == "__main__":
    # 回测策略
    symbol = "AAPL"
    start_date = "2020-01-01"
    end_date = "2024-01-01"
    
    results = backtest_strategy(symbol, start_date, end_date)
    
    # 打印性能指标
    print(f"策略回测结果 ({symbol})")
    print(f"总收益率: {results['total_return']:.2f}%")
    print(f"夏普比率: {results['sharpe_ratio']:.2f}")
    print(f"最大回撤: {results['max_drawdown']:.2f}%")
    print(f"交易次数: {len(results['trades'])}")
    
    # 绘制图表
    plot_results(results, symbol)